In [6]:
from transformers import pipeline
from datasets import load_dataset
import evaluate
import pandas as pd

summarizer = pipeline("summarization", model="facebook/bart-large-cnn")
dataset = load_dataset("cnn_dailymail", "3.0.0", split="test[:5]")

print("Number of articles:", len(dataset))
print(dataset[0]["article"][:500])

Device set to use mps:0


Number of articles: 5
(CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alleged crimes in Palestinian territories. The formal accession was marked with a ceremony at The Hague, in the Netherlands, where the court is based. The Palestinians signed the ICC's founding Rome Statute in January, when they also accepted its jurisdiction over alleged crimes committed "in the occupied Palestinian territory, includin


In [10]:
generated = []
reference = []

for sample in dataset:
    article = sample["article"]

    result = summarizer(article[:3000])

    generated.append(result[0]["summary_text"])
    reference.append(sample["highlights"])
    

print("Generated summaries:", len(generated))

Generated summaries: 5


In [11]:
rouge = evaluate.load("rouge")

scores = rouge.compute(
    predictions=generated,
    references=reference
)

print("ROUGE-1:", scores["rouge1"])
print("ROUGE-2:", scores["rouge2"])
print("ROUGE-L:", scores["rougeL"])
print("ROUGE-Lsum:", scores["rougeLsum"])

ROUGE-1: 0.44037313676245143
ROUGE-2: 0.23211233211233212
ROUGE-L: 0.32165215216847687
ROUGE-Lsum: 0.3850353892821031


In [12]:
df = pd.DataFrame({
    "Reference Summary": reference,
    "Generated Summary": generated
})

display(df.head())
df.to_csv("generated_summaries.csv", index=False)

print("CSV saved successfully!")

,Reference Summary,Generated Summary
0,Membership gives the ICC jurisdiction over all...,The Palestinian Authority becomes the 123rd me...
1,"Theia, a bully breed mix, was apparently hit b...","Theia, a one-year-old bully breed mix, was hit..."
2,Mohammad Javad Zarif has spent more time with ...,Mohammad Javad Zarif is the Iranian foreign mi...
3,17 Americans were exposed to the Ebola virus w...,The five were exposed to Ebola in Sierra Leone...
4,Student is no longer on Duke University campus...,The noose was found on a tree near a student u...


CSV saved successfully!
